# 01. Deep Research 模式

Deep Research = RAG 的主动版。单轮 RAG 是「问一次查一次」；deep research 是 agent 自主的多轮循环：

```
plan（拆子问题）→ search（逐个子问题检索）→ synthesize（综合 + 引用）→ verify（覆盖度检查）→ 不足则再一轮
```

与单轮 RAG 的三点本质差异：
1. **多轮**：根据已读内容决定下一步查什么
2. **自主**：子问题由 agent 生成，不是用户给定的
3. **可验证**：综合后有显式的覆盖度检查环节

> **检查点**：能解释为什么 deep research 的报告必须有引用 — 没有引用就无法做「引用准确性」评估，无法闭环。


In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'xmake.lua').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'agents'))

from scripts.nb_helpers import setup, chat, chat_json, chat_with_tools, show_trace
client = setup()


🌐 API 模式 — model=deepseek-v4-flash  base_url=https://api.deepseek.com


In [2]:
from pathlib import Path
import subprocess, sys

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'xmake.lua').exists():
    ROOT = ROOT.parent

SCRIPT = ROOT / 'agents' / 'deep-research' / 'scripts' / 'mini_deep_researcher.py'
result = subprocess.run([sys.executable, str(SCRIPT), '--verbose'], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr, file=sys.stderr)


=== Mini Deep Researcher (toy) ===
Question: agent 记忆系统有哪些主流方案，如何选型

Plan (3 sub-questions):
  - agent 记忆系统有哪些主流方案，如何选型
  - 主流方案对比
  - 评估与选型建议

[search] agent 记忆系统有哪些主流方案，如何选型
  [D5] 向量数据库选型
  [D6] 记忆的评估

[search] 主流方案对比
  [D6] 记忆的评估
  [D1] Mem0

[search] 评估与选型建议
  [D5] 向量数据库选型
  [D6] 记忆的评估

[round 2] coverage follow-up → ['D5', 'D6']

=== Report ===
# 研究报告：agent 记忆系统有哪些主流方案，如何选型

## agent 记忆系统有哪些主流方案，如何选型
- [D5] agent 记忆的存储层常用 pgvector（已有 Postgres 时零新增运维）、Qdrant（Rust 高性能、...
- [D6] LoCoMo 与 LongMemEval 是 agent 记忆的主流基准。评估维度：事实抽取准确率、时序推理（事实何时失...
## 主流方案对比
- [D6] LoCoMo 与 LongMemEval 是 agent 记忆的主流基准。评估维度：事实抽取准确率、时序推理（事实何时失...
- [D1] Mem0 是开源的 agent 记忆层（YC S24）。ADD-only 提取管线：每轮对话提取事实并存入向量库与图存储...
## 评估与选型建议
- [D5] agent 记忆的存储层常用 pgvector（已有 Postgres 时零新增运维）、Qdrant（Rust 高性能、...
- [D6] LoCoMo 与 LongMemEval 是 agent 记忆的主流基准。评估维度：事实抽取准确率、时序推理（事实何时失...
## 补充检索：评估与选型
- [D5] agent 记忆的存储层常用 pgvector（已有 Postgres 时零新增运维）、Qdrant（Rust 高性能、...
- [D6] LoCoMo 与 LongMemEval 是 agent 记忆的主流基准。评估维度：事实抽取准确率、时

## 研究循环的工程要点

| 环节 | 关键设计 | 失败模式 |
|------|---------|---------|
| plan | 子问题递进（是什么 → 对比 → 选型） | 子问题互相重叠，浪费检索 |
| search | 每个子问题独立检索 top-k | 一次性大 query 检索，噪声大 |
| synthesize | 强制 [来源] 标注 | 模型混入材料外的「知识」→ 幻觉 |
| verify | 独立评审调用检查覆盖度 | 自评自夸（同一 prompt 上下文里自评） |


In [3]:
from pathlib import Path
import subprocess, sys

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'xmake.lua').exists():
    ROOT = ROOT.parent

SCRIPT = ROOT / 'agents' / 'deep-research' / 'scripts' / 'mini_deep_researcher.py'
result = subprocess.run([sys.executable, str(SCRIPT), '--use-api', '--verbose'], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr, file=sys.stderr)


=== Mini Deep Researcher (API) ===
Question: agent 记忆系统有哪些主流方案，如何选型

Plan (3 sub-questions):
  - 当前主流的 agent 记忆系统方案有哪些？
  - 这些方案在存储机制、检索效率、可扩展性等方面有哪些优缺点？
  - 如何根据 agent 的应用场景、性能需求和资源约束进行记忆系统的选型决策？

[search] 当前主流的 agent 记忆系统方案有哪些？
  [D7] 工作记忆与上下文工程
  [D4] RAG 与记忆的区别

[search] 这些方案在存储机制、检索效率、可扩展性等方面有哪些优缺点？
  [D6] 记忆的评估
  [D5] 向量数据库选型

[search] 如何根据 agent 的应用场景、性能需求和资源约束进行记忆系统的选型决策？
  [D5] 向量数据库选型
  [D4] RAG 与记忆的区别

[round 2] coverage follow-up → ['D5', 'D6']

=== Report ===
Agent记忆系统主流方案采用双层架构：**工作记忆**管理当前上下文窗（通过压缩、隔离、缓存），**长期记忆**跨会话持久化（代表方案如Mem0/Zep），两者皆需且瓶颈通常在工作记忆[D7]。存储层选型：pgvector（已有Postgres时零运维）、Qdrant（高性能边缘部署）、Milvus（十亿级分布式）、Chroma（原型），应优先评估规模与现有基础设施[D5]。检索需结合BM25、实体与时序维度，仅靠dense搜索不足，评估基准推荐LoCoMo与LongMemEval[D6]。生产系统还需区分RAG（静态知识）与记忆（交互历史），两者互补[D4]。

**未覆盖**：具体缓存策略实现、记忆压缩/遗忘机制、分布式一致性细节、成本对比。

=== Coverage check ===
评分：7/10  
缺失方面：未讨论记忆的遗忘机制、记忆合并/摘要策略以及分布式一致性保证的具体实现。

Corpus: 8 docs | sub-questions: 4 | mode: API
Takeaway: deep research = RAG 的主动版 — agent 自己决定查什么、查几轮、何时停。



## 真实研究循环观察

API 模式下注意两点：(1) LLM 拆的子问题与 toy 的关键词拆分差异；(2) 报告末尾的「未覆盖」部分 —
这是 verify 环节的输入，驱动下一轮检索。

## 练习

1. 把 `retrieve` 换成真实搜索 API（如 Tavily/SerpAPI，需自备 key），对比本地语料与真实网页的报告质量。
2. 实现第二轮由 verify 驱动：把「未覆盖」方面作为新一轮子问题，而不是脚本里写死的 follow_up。
3. 给 corpus 加 3 篇互相矛盾的文档，观察 synthesize 如何处理冲突（提示：好报告应呈现分歧而非选一个）。
